## Lightgbm の OOF を読み込む

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score

In [2]:
ROOT = Path.cwd()

if ROOT.name == "smartphone_addiction":
    ROOT = ROOT.parent.parent

OUTPUT_DIR = ROOT / "output" / "smartphone_addiction"

oof_lgb = np.load(
    OUTPUT_DIR / "oof_lightgbm.npy"
)

test_pred_lgb = np.load(
    OUTPUT_DIR / "test_pred_lightgbm.npy"
)

## CatBoost の OOF も読み込む

In [3]:
# データの読み込み
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier

DATA_DIR = (
    ROOT
    / "data"
    / "smartphone_addiction"
    / "raw"
)

train = pd.read_csv(
    DATA_DIR / "train.csv"
)

test = pd.read_csv(
    DATA_DIR / "test.csv"
)

TARGET = "addicted_label"

X_cb = train.drop(
    columns=[TARGET, "id"]
).copy()

y = train[TARGET].copy()

X_test_cb = test.drop(
    columns=["id"]
).copy()

In [4]:
# カテゴリ列
categorical_cols = (
    X_cb
    .select_dtypes(
        include=["object", "category"]
    )
    .columns
    .tolist()
)

for col in categorical_cols:

    X_cb[col] = (
        X_cb[col]
        .fillna("Missing")
        .astype(str)
    )

    X_test_cb[col] = (
        X_test_cb[col]
        .fillna("Missing")
        .astype(str)
    )

C:\Users\Owner\AppData\Local\Temp\ipykernel_22252\2962434890.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  .select_dtypes(


In [5]:
# tuned catboost のパラメータ
best_params = {
    "depth": 6,
    "l2_leaf_reg": 5.0,
    "random_strength": 0.5,
    "bagging_temperature": 0.0
}

In [6]:
# 3-fold
cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

oof_cb = np.zeros(
    len(X_cb)
)

test_pred_cb = np.zeros(
    len(X_test_cb)
)

fold_scores_cb = []
best_iterations_cb = []

In [7]:
for fold, (
    train_idx,
    valid_idx
) in enumerate(
    cv.split(X_cb, y),
    start=1
):

    print(
        f"\n===== Fold {fold} ====="
    )

    X_train = X_cb.iloc[
        train_idx
    ]

    X_valid = X_cb.iloc[
        valid_idx
    ]

    y_train = y.iloc[
        train_idx
    ]

    y_valid = y.iloc[
        valid_idx
    ]

    model = CatBoostClassifier(
        iterations=7000,
        learning_rate=0.05,

        loss_function="Logloss",
        eval_metric="AUC",

        random_seed=42,
        verbose=500,
        allow_writing_files=False,

        **best_params
    )

    model.fit(
        X_train,
        y_train,

        cat_features=
            categorical_cols,

        eval_set=(
            X_valid,
            y_valid
        ),

        early_stopping_rounds=200
    )

    valid_pred = (
        model.predict_proba(
            X_valid
        )[:, 1]
    )

    oof_cb[
        valid_idx
    ] = valid_pred

    test_pred_cb += (
        model.predict_proba(
            X_test_cb
        )[:, 1]
        / cv.n_splits
    )

    fold_auc = roc_auc_score(
        y_valid,
        valid_pred
    )

    fold_scores_cb.append(
        fold_auc
    )

    best_iterations_cb.append(
        model.get_best_iteration()
    )

    print(
        "AUC:",
        fold_auc
    )


===== Fold 1 =====
0:	test: 0.9055699	best: 0.9055699 (0)	total: 348ms	remaining: 40m 33s
500:	test: 0.9509404	best: 0.9509404 (500)	total: 1m 22s	remaining: 17m 45s
1000:	test: 0.9566083	best: 0.9566083 (1000)	total: 2m 42s	remaining: 16m 12s
1500:	test: 0.9594375	best: 0.9594375 (1500)	total: 4m 4s	remaining: 14m 56s
2000:	test: 0.9608339	best: 0.9608339 (2000)	total: 5m 27s	remaining: 13m 38s
2500:	test: 0.9615864	best: 0.9615864 (2500)	total: 6m 52s	remaining: 12m 21s
3000:	test: 0.9620479	best: 0.9620479 (3000)	total: 8m 16s	remaining: 11m 1s
3500:	test: 0.9624016	best: 0.9624016 (3500)	total: 9m 40s	remaining: 9m 39s
4000:	test: 0.9626159	best: 0.9626160 (3999)	total: 11m 4s	remaining: 8m 17s
4500:	test: 0.9627749	best: 0.9627752 (4496)	total: 12m 28s	remaining: 6m 55s
5000:	test: 0.9629104	best: 0.9629120 (4991)	total: 13m 52s	remaining: 5m 32s
5500:	test: 0.9629992	best: 0.9629995 (5499)	total: 15m 16s	remaining: 4m 9s
6000:	test: 0.9630862	best: 0.9630869 (5995)	total: 16m 40

In [8]:
# 保存
np.save(
    OUTPUT_DIR / "oof_catboost_tuned.npy",
    oof_cb
)

np.save(
    OUTPUT_DIR / "test_pred_catboost_tuned.npy",
    test_pred_cb
)

## 予測相関を確認

In [9]:
prediction_corr = np.corrcoef(
    oof_cb,
    oof_lgb
)[0, 1]

print(
    "Prediction correlation:",
    prediction_corr
)

Prediction correlation: 0.9927683309410137


In [10]:
# OOF_AUC も確認
auc_cb = roc_auc_score(
    y,
    oof_cb
)

auc_lgb = roc_auc_score(
    y,
    oof_lgb
)

print(
    "CatBoost OOF:",
    auc_cb
)

print(
    "LightGBM OOF:",
    auc_lgb
)

CatBoost OOF: 0.9636859867800479
LightGBM OOF: 0.9636470878336719


## ensemble

In [11]:
# 50:50
oof_blend_50 = (
    0.5 * oof_cb
    +
    0.5 * oof_lgb
)

auc_blend_50 = roc_auc_score(
    y,
    oof_blend_50
)

print(
    "CatBoost:",
    auc_cb
)

print(
    "LightGBM:",
    auc_lgb
)

print(
    "50:50 Ensemble:",
    auc_blend_50
)

CatBoost: 0.9636859867800479
LightGBM: 0.9636470878336719
50:50 Ensemble: 0.9642010708352016


In [12]:
# 重み変えてみる
blend_results = []

for cb_weight in np.arange(
    0.0,
    1.01,
    0.1
):

    lgb_weight = (
        1.0 - cb_weight
    )

    blended_pred = (
        cb_weight
        * oof_cb
        +
        lgb_weight
        * oof_lgb
    )

    auc = roc_auc_score(
        y,
        blended_pred
    )

    blend_results.append({
        "catboost_weight":
            cb_weight,

        "lightgbm_weight":
            lgb_weight,

        "auc":
            auc
    })

blend_df = pd.DataFrame(
    blend_results
)

display(
    blend_df.sort_values(
        "auc",
        ascending=False
    )
)

,catboost_weight,lightgbm_weight,auc
5,0.5,0.5,0.964201
6,0.6,0.4,0.964184
4,0.4,0.6,0.964176
7,0.7,0.3,0.964125
3,0.3,0.7,0.964109
8,0.8,0.2,0.964023
2,0.2,0.8,0.964000
9,0.9,0.1,0.963877
1,0.1,0.9,0.963847
10,1.0,0.0,0.963686


In [13]:
# 3つのfoldでも安定して改善しているか
cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

fold_ensemble_results = []

for fold, (_, valid_idx) in enumerate(
    cv.split(X_cb, y),
    start=1
):
    y_valid = y.iloc[valid_idx]

    pred_cb = oof_cb[valid_idx]
    pred_lgb = oof_lgb[valid_idx]

    pred_ensemble = (
        0.5 * pred_cb
        + 0.5 * pred_lgb
    )

    auc_cb_fold = roc_auc_score(
        y_valid,
        pred_cb
    )

    auc_lgb_fold = roc_auc_score(
        y_valid,
        pred_lgb
    )

    auc_ensemble_fold = roc_auc_score(
        y_valid,
        pred_ensemble
    )

    fold_ensemble_results.append({
        "fold": fold,
        "CatBoost": auc_cb_fold,
        "LightGBM": auc_lgb_fold,
        "Ensemble": auc_ensemble_fold
    })

fold_ensemble_df = pd.DataFrame(
    fold_ensemble_results
)

display(fold_ensemble_df)

,fold,CatBoost,LightGBM,Ensemble
0,1,0.963209,0.963158,0.963733
1,2,0.963933,0.963979,0.964483
2,3,0.963918,0.963814,0.964391


## submision

In [14]:
test_pred_ensemble = (
    0.5 * test_pred_cb
    + 0.5 * test_pred_lgb
)

In [15]:
sample_submission = pd.read_csv(
    DATA_DIR / "sample_submission.csv"
)

submission = sample_submission.copy()

submission[TARGET] = (
    test_pred_ensemble
)

assert submission["id"].equals(
    test["id"]
)

assert submission[TARGET].notna().all()

assert submission[TARGET].between(
    0, 1
).all()

submission.to_csv(
    OUTPUT_DIR
    / "submission_catboost_lightgbm_5050.csv",
    index=False
)